# Market basket analysis with Eclat on Google Colab

A retailer wants to know which products customers buy together. This notebook takes the receipts of Online Retail II, mines them with the team's Eclat, derives association rules, and shows the rules with the highest lift. Run all cells on a CPU runtime. Colab has no Julia kernel, so Python cells install Julia 1.13.2, check its SHA-256, and call it.

The cells run in this order: install Julia and fetch the repository at an exact commit, instantiate the project, run the package tests (they include a seven-basket case whose support, confidence, and lift are worked out by hand), download and clean the data (about 95 MB), mine and derive the rules (about three minutes), show the tables, and pack the evidence. The whole run takes about ten minutes.

Set `ITEMSET_REVISION` in the first code cell to a full commit SHA to replay a run. The data and the results stay in `/content`, outside Git. `applications/market_basket/README.md` explains each cleaning step, and `docs/market-basket-rules-2026-10-29.md` records the results.

In [ ]:
import hashlib
import os
import re
import subprocess
import tarfile
import urllib.request
from pathlib import Path

REPOSITORY = "https://github.com/NgTHung/Itemset"
ITEMSET_REVISION = ""  # A full commit SHA replays a run; empty takes the head of the default branch.
JULIA_URL = "https://julialang-s3.julialang.org/bin/linux/x64/1.13/julia-1.13.2-linux-x86_64.tar.gz"
JULIA_SHA256 = "2ecdfc9890ab0a565fdd92404c915ba2a04a933dbd5d292cc224d13819d17d07"

root, julia_home = Path("/content/Itemset-market-basket"), Path("/content/julia-1.13.2")
revision = ITEMSET_REVISION or os.environ.get("ITEMSET_REVISION") or subprocess.check_output(
    ["git", "ls-remote", "--exit-code", REPOSITORY, "HEAD"], text=True, timeout=60).split()[0]
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError(f"ITEMSET_REVISION must be a full lowercase 40-character commit SHA, got {revision!r}")
if not root.exists():
    subprocess.run(["git", "clone", "--", REPOSITORY, str(root)], check=True)
subprocess.run(["git", "-C", str(root), "checkout", "--detach", revision], check=True)

if not julia_home.exists():
    archive, digest = Path("/content/julia-1.13.2.tar.gz"), hashlib.sha256()
    with urllib.request.urlopen(JULIA_URL, timeout=60) as source, archive.open("wb") as target:
        while chunk := source.read(1024 * 1024):
            target.write(chunk)
            digest.update(chunk)
    if digest.hexdigest() != JULIA_SHA256:
        archive.unlink()
        raise ValueError(f"Julia archive SHA-256 is {digest.hexdigest()}, expected {JULIA_SHA256}")
    julia_home.mkdir()
    with tarfile.open(archive) as package:
        package.extractall(julia_home, filter="data")
os.environ["PATH"] = f"{next(julia_home.glob('julia-*/bin'))}{os.pathsep}{os.environ['PATH']}"
os.environ["JULIA_NUM_THREADS"] = "1"
os.chdir(root)


def julia(*args):
    """Run Julia in the repository and stop the notebook on a nonzero exit, which a shell escape would not."""
    process = subprocess.Popen(["julia", "--startup-file=no", "--project=.", *args], stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError(f"julia {' '.join(args)} exited with {process.returncode}")


print("Revision:", revision)
print(subprocess.check_output(["julia", "--version"], text=True).strip())

In [ ]:
julia("-e", "using Pkg; Pkg.instantiate()")

In [ ]:
# The suite compares the rules with a direct count on random databases and checks the hand-worked case.
julia("test/runtests.jl")

In [ ]:
# Downloads Online Retail II, checks its SHA-256, and writes one basket per invoice to data/raw/market_basket/.
julia("applications/market_basket/prepare.jl")

In [ ]:
# Mines the baskets at minsup 0.01 and keeps rules with confidence of at least 0.5 (about three minutes).
julia("applications/market_basket/analyze.jl")

## Reading the tables

Each rule X => Y reads: customers who buy X also buy Y. Support counts the baskets that hold X and Y together. Confidence is the share of baskets with X that also hold Y. Lift divides that confidence by the share of all baskets that hold Y, so a lift above 1 means X and Y sell together more than chance. The measures are exact fractions printed to four decimals.

A product group of m items yields up to 2^m - 2 rules, so `top_groups.tsv` lists the best rule of each of the first ten distinct groups next to the ten best rules.

In [ ]:
import csv

import pandas as pd
from IPython.display import display

results = root / "results/raw/market_basket"
for name in ("top_rules.tsv", "top_groups.tsv"):
    print(name)
    display(pd.read_csv(results / name, sep="\t", quoting=csv.QUOTE_NONE).drop(columns=["antecedent_ids", "consequent_ids"]))
print("summary.toml")
print((results / "summary.toml").read_text(encoding="utf-8"))

In [ ]:
import json
import zipfile

# Files go in by relative path, so the ZIP holds the same names whatever the runtime. The basket file is data and stays out.
results = root / "results/raw/market_basket"
files = sorted(p for p in results.rglob("*") if p.is_file()) + [root / "data/raw/market_basket/cleaning.tsv"]
record = {
    "repository": REPOSITORY,
    "revision": revision,
    "julia": subprocess.check_output(["julia", "--version"], text=True).strip(),
    "julia_archive_sha256": JULIA_SHA256,
    "sha256": {p.relative_to(root).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in files},
}
evidence = Path("/content/itemset-market-basket-evidence.zip")
with zipfile.ZipFile(evidence, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.relative_to(root))
    archive.writestr("result.json", json.dumps(record, indent=2))
print(f"Evidence: {evidence} ({len(files)} files). Download it from the Files sidebar.")